<h3>형식에 맞는 문서 단위</h3>
<p><strong>개념</strong> DOCX는 문단과 표, HTML은 본문 영역, CSV는 행이 기본 단위입니다. 하나의 범용 문자열 변환보다 형식별 구조를 먼저 읽습니다.</p>

In [1]:
# %pip은 현재 커널에 HTML과 DOCX 로더를 설치합니다. -q는 설치 로그를 줄입니다.
# 버전을 고정해 예제에서 사용하는 클래스와 메서드를 동일하게 맞춥니다.
%pip install -q "beautifulsoup4==4.15.0" "python-docx==1.2.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 형식마다 전용 로더를 가져오고, 현재 위치의 상위 폴더에서 실습 루트를 찾습니다.
# fixtures는 세 로더에 전달할 작은 실제 파일이 있는 폴더입니다.
from pathlib import Path
from bs4 import BeautifulSoup
from docx import Document
import csv

roots = [Path.cwd(), *Path.cwd().parents]
labs = [p / "labs/part3/ch10-document-processing" for p in roots
        if (p / "labs/part3/ch10-document-processing").is_dir()]
if not labs: raise FileNotFoundError("저장소 루트에서 노트북을 실행하세요")
lab = labs[0]
fixtures = lab / "fixtures"


<h3>세 형식을 짧게 읽기</h3>
<p><strong>확인할 점</strong> 각 결과가 어떤 단위를 보존하는지 출력에서 확인합니다.</p>

In [3]:
# DOCX 문단, HTML 본문, CSV 행을 각각 해당 라이브러리의 반환 형태로 읽습니다.
# 리스트 컴프리헨션은 조건을 만족하는 항목만 모으며 decompose()는 태그를 제거합니다.
docx = Document(fixtures / "sample.docx")
paragraphs = [p.text for p in docx.paragraphs if p.text.strip()]
soup = BeautifulSoup((fixtures / "sample.html").read_text(), "html.parser")
for tag in soup(["script", "style"]): tag.decompose()
with (fixtures / "sample.csv").open(encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))
print({"docx_paragraphs": len(paragraphs),
       "html": soup.get_text(" ", strip=True)[:50], "csv_rows": len(rows)})


{'docx_paragraphs': 2, 'html': '반품 안내 수령 후 7일 이내 신청합니다.', 'csv_rows': 2}


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 존재하지 않는 CSV 열을 읽으면 스키마 불일치를 즉시 확인할 수 있습니다.</p>

In [4]:
# 존재하지 않는 열 이름으로 CSV 스키마 불일치를 재현합니다.
# 딕셔너리에 없는 키를 대괄호로 읽으면 KeyError가 발생합니다.
try:
    print(rows[0]["없는_열"])
except KeyError as error:
    print(f"caught: CSV schema mismatch {error}")


caught: CSV schema mismatch '없는_열'


<p><strong>수정 이유</strong> 필요한 열을 먼저 검증한 뒤 실제 열 이름으로 값을 읽습니다.</p>

In [5]:
# 필요한 열 집합이 실제 열 집합의 부분집합인지 먼저 검증합니다.
# sorted()로 출력 순서를 고정하고, 딕셔너리 컴프리헨션으로 필요한 값만 반환합니다.
required = {"상품", "보관"}
available = set(rows[0])
assert required <= available, f"missing columns: {required - available}"
print({key: rows[0][key] for key in sorted(required)})


{'보관': '실온', '상품': '쿠키'}
